# One-vs-Rest Logistic Regression with Thresholding

This notebook builds the threshold-based one-vs-rest (OVR) logistic regression competitor to the cosine-similarity threshold model.

## Plan
- Load labeled Accountancy job data and the corresponding train/validation embeddings.
- Build a multilabel target matrix where each skill is treated as its own binary classification problem.
- Train one logistic regression classifier per skill using the job embeddings as input features.
- Convert predicted probabilities into skill labels using thresholding.
- Compare thresholding strategies such as a single global threshold and skill-specific thresholds.
- Evaluate with the same multilabel metrics used in the cosine-threshold notebook: micro/macro precision, recall, and F1.

## Data Used
- Labels: `../data/acc/audit_tax_accounting_jobs.csv`
- Skill universe: `../embedding/acc/acc_skills_embeddings.jsonl`
- Job embeddings: `../embedding/acc/acc_jobs_embeddings.jsonl`

This notebook focuses on the threshold version of OVR logistic regression, so the prediction rule is:

`P(skill_j = 1 | job_embedding) >= threshold_j`

which is the logistic-regression analogue of the cosine-threshold rule:

`cosine(job_embedding, skill_embedding_j) >= threshold_j`


## ELI5: What This Notebook Is Doing

Think of each job as a bundle of numbers called an embedding. We want to use those numbers to guess which skills belong to that job.

### What is OVR?
OVR stands for one-vs-rest.

That means we train one yes/no model for each skill:
- `Accounting Standards`: yes or no?
- `Financial Reporting`: yes or no?
- `Tax Compliance`: yes or no?

So if there are many skills, we train many small binary models instead of one big model.

### How our data fits in
- The CSV gives the true answers: which skills each job already has.
- The job embeddings file gives the input features for training and validation.
- The skill embeddings file gives the full ACC skill list we are predicting over.

### What the workflow looks like
1. Read jobs and their true skill labels.
2. Turn those labels into a binary matrix.
   Example: if a job has two skills, those two columns become 1 and all others become 0.
3. Train one logistic regression model per skill.
4. Each model outputs a probability, like `0.82` for "this job has Financial Reporting".
5. Apply a threshold to turn that probability into a final yes/no prediction.
6. Compare predicted skills against actual skills using precision, recall, and F1.

### Why thresholding matters
A probability is not yet a final label. We still need to decide what counts as high enough.

For example:
- if threshold = `0.50`, then `0.82` becomes yes
- if threshold = `0.90`, then `0.82` becomes no

This notebook is specifically for the threshold version of OVR, so the core rule is:

`predict skill j if P(skill_j = 1 | job_embedding) >= threshold_j`

### What outputs we want
Best case, we want three main outputs:
- an overall metrics table with strong micro-F1 and decent macro-F1
- a threshold table showing the cutoff used for each skill
- a predictions table showing actual skills vs predicted skills for each job

### What success looks like
The ideal outcome is that this OVR-threshold model performs as well as or better than the cosine-similarity-threshold model on the same validation setup.

In simple terms, best case means:
- better micro-F1
- fewer false positives
- more sensible per-skill decision boundaries


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

# Data paths
JOB_LABEL_CSV = Path("../data/acc/audit_tax_accounting_jobs.csv")
EMBEDDINGS_PATH = Path("../embedding/acc")

# Load ACC embedding files
jobs_embeddings = pd.read_json(EMBEDDINGS_PATH / "acc_jobs_embeddings.jsonl", lines=True)
skills_embeddings = pd.read_json(EMBEDDINGS_PATH / "acc_skills_embeddings.jsonl", lines=True)

# Split jobs into train / validation using the built-in split column
jobs_train_embeddings = jobs_embeddings.loc[jobs_embeddings["split"] == "train"].reset_index(drop=True)
jobs_val_embeddings = jobs_embeddings.loc[jobs_embeddings["split"] == "val"].reset_index(drop=True)

print("Loaded ACC embeddings:")
print(f"- jobs total: {len(jobs_embeddings)}")
print(f"- jobs train: {len(jobs_train_embeddings)}")
print(f"- jobs val: {len(jobs_val_embeddings)}")
print(f"- skills total: {len(skills_embeddings)}")


Loaded ACC embeddings:
- jobs total: 98
- jobs train: 59
- jobs val: 19
- skills total: 229


In [ ]:
def parse_skill_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return []

    seen = set()
    skills = []
    for raw in text.split("|"):
        skill = raw.strip()
        if skill and skill not in seen:
            seen.add(skill)
            skills.append(skill)
    return skills


job_labels = pd.read_csv(JOB_LABEL_CSV, usecols=["uuid", "title", "extracted_skills"])

jobs_train_df = jobs_train_embeddings.merge(
    job_labels,
    left_on="job_id",
    right_on="uuid",
    how="left",
    suffixes=("_embed", "_label"),
)
jobs_val_df = jobs_val_embeddings.merge(
    job_labels,
    left_on="job_id",
    right_on="uuid",
    how="left",
    suffixes=("_embed", "_label"),
)

for name, df in [("jobs_train_df", jobs_train_df), ("jobs_val_df", jobs_val_df)]:
    missing = int(df["uuid"].isna().sum())
    if missing:
        raise ValueError(f"{name} has {missing} embeddings without matching labels")

jobs_train_df["actual_skill_lists"] = jobs_train_df["extracted_skills"].map(parse_skill_list)
jobs_val_df["actual_skill_lists"] = jobs_val_df["extracted_skills"].map(parse_skill_list)

print("Join checks passed:")
print(f"- jobs train labeled rows: {len(jobs_train_df)}")
print(f"- jobs val labeled rows: {len(jobs_val_df)}")
print()
print("Preview:")
print(jobs_train_df[["job_id", "title_label", "extracted_skills"]].head(3).to_string(index=False))

Join checks passed:
- jobs train labeled rows: 59
- jobs val labeled rows: 19

Preview:
                          job_id                                                      title_label                                                                                                                                                                                                                                                                                                                                                                                  extracted_skills
da824c9b2923bf8768966ce83cafa56e 6723 - Accountant [ERP System | Exp in Audit Support | Clementi] Cash Flow Management | Accounting Standards | Financial Analysis | Financial Transactions | Partnership Management | Management Decision Making | Transactional Accounting | Regulatory Compliance | Financial Reporting | Tax Compliance | Financial Statements Review | Internal Controls | Audit Compliance | Financial Closing | Accou

In [ ]:
def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)

    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            if skill in skill_to_idx:
                indicator[row_idx, skill_to_idx[skill]] = 1

    return indicator

# Skill universe
skill_names = skills_embeddings["skill_name"].tolist()

# Feature matrices
X_train = np.vstack(jobs_train_df["embedding"].to_numpy()).astype(np.float32)
X_val = np.vstack(jobs_val_df["embedding"].to_numpy()).astype(np.float32)

# Label matrices
Y_train = build_indicator_matrix(jobs_train_df["actual_skill_lists"].tolist(), skill_names)
Y_val = build_indicator_matrix(jobs_val_df["actual_skill_lists"].tolist(), skill_names)

print("Shapes:")
print(f"X_train: {X_train.shape}")
print(f"Y_train: {Y_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"Y_val:   {Y_val.shape}")
print(f"num_skills: {len(skill_names)}")

if X_train.shape[0] != Y_train.shape[0]:
    raise ValueError("X_train and Y_train row counts do not match")

if X_val.shape[0] != Y_val.shape[0]:
    raise ValueError("X_val and Y_val row counts do not match")

if Y_train.shape[1] != len(skill_names) or Y_val.shape[1] != len(skill_names):
    raise ValueError("Label matrix column count does not match number of skills")

Shapes:
X_train: (59, 2560)
Y_train: (59, 229)
X_val:   (19, 2560)
Y_val:   (19, 229)
num_skills: 229


In [ ]:
train_proba = np.zeros((X_train.shape[0], len(skill_names)), dtype=np.float32)
val_proba = np.zeros((X_val.shape[0], len(skill_names)), dtype=np.float32)

fitted_models = {}
skipped_skills = []

for j, skill_name in enumerate(skill_names):
    y_train_j = Y_train[:, j]

    # Logistic regression cannot train if only one class is present
    if len(np.unique(y_train_j)) < 2:
        skipped_skills.append(skill_name)
        continue

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )
    model.fit(X_train, y_train_j)

    train_proba[:, j] = model.predict_proba(X_train)[:, 1]
    val_proba[:, j] = model.predict_proba(X_val)[:, 1]

    fitted_models[skill_name] = model

print("OVR training complete.")
print(f"train_proba shape: {train_proba.shape}")
print(f"val_proba shape:   {val_proba.shape}")
print(f"fitted models:     {len(fitted_models)}")
print(f"skipped skills:    {len(skipped_skills)}")

if skipped_skills:
    print("\nFirst few skipped skills:")
    print(skipped_skills[:10])

OVR training complete.
train_proba shape: (59, 229)
val_proba shape:   (19, 229)
fitted models:     52
skipped skills:    177

First few skipped skills:
['Account Management', 'Analytics and Computational Modelling', 'Asset and Liability Management', 'Attribution Analysis', 'Audit Frameworks', 'Auditor Independence', 'Behavioural Finance', 'Benchmarking', 'Big Data Analytics', 'Block Trading']


In [ ]:
def micro_f1_from_predictions(y_true, y_pred):
    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1

candidate_thresholds = np.unique(val_proba)
candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

best_threshold = None
best_metrics = None

for threshold in candidate_thresholds:
    y_pred = (val_proba >= threshold).astype(np.uint8)
    precision, recall, f1 = micro_f1_from_predictions(Y_val, y_pred)

    candidate = (f1, recall, -threshold)
    if best_metrics is None or candidate > best_metrics:
        best_metrics = candidate
        best_threshold = float(threshold)

global_val_pred = (val_proba >= best_threshold).astype(np.uint8)
global_precision, global_recall, global_f1 = micro_f1_from_predictions(Y_val, global_val_pred)

print("Best global threshold on validation:")
print(f"threshold: {best_threshold:.4f}")
print(f"micro_precision: {global_precision:.4f}")
print(f"micro_recall:    {global_recall:.4f}")
print(f"micro_f1:        {global_f1:.4f}")

Best global threshold on validation:
threshold: 0.5056
micro_precision: 0.7007
micro_recall:    0.8276
micro_f1:        0.7589


In [ ]:
def tune_best_threshold_for_one_skill(y_true, y_score):
    candidate_thresholds = np.unique(y_score)
    candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

    best_threshold = 0.5
    best_candidate = None

    for threshold in candidate_thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)

        tp = np.logical_and(y_pred == 1, y_true == 1).sum()
        fp = np.logical_and(y_pred == 1, y_true == 0).sum()
        fn = np.logical_and(y_pred == 0, y_true == 1).sum()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

        candidate = (f1, recall, -threshold)
        if best_candidate is None or candidate > best_candidate:
            best_candidate = candidate
            best_threshold = float(threshold)

    return best_threshold

skill_thresholds = np.full(len(skill_names), 0.5, dtype=np.float32)

for j, skill_name in enumerate(skill_names):
    if skill_name in fitted_models:
        skill_thresholds[j] = tune_best_threshold_for_one_skill(
            Y_val[:, j],
            val_proba[:, j],
        )

skill_val_pred = (val_proba >= skill_thresholds[np.newaxis, :]).astype(np.uint8)
skill_precision, skill_recall, skill_f1 = micro_f1_from_predictions(Y_val, skill_val_pred)

thresholds_df = pd.DataFrame({
    "skill_name": skill_names,
    "threshold": skill_thresholds,
    "was_fitted": [skill in fitted_models for skill in skill_names],
    "train_positive_support": Y_train.sum(axis=0),
    "val_positive_support": Y_val.sum(axis=0),
})

print("Skill-specific thresholds on validation:")
print(f"micro_precision: {skill_precision:.4f}")
print(f"micro_recall:    {skill_recall:.4f}")
print(f"micro_f1:        {skill_f1:.4f}")

print("\nThreshold preview:")
print(thresholds_df.head(10).to_string(index=False))

Skill-specific thresholds on validation:
micro_precision: 0.1536
micro_recall:    0.9310
micro_f1:        0.2637

Threshold preview:
                           skill_name  threshold  was_fitted  train_positive_support  val_positive_support
                   Account Management   0.500000       False                       0                     0
                 Accounting Standards   0.525089        True                      42                    13
           Accounting and Tax Systems   0.339532        True                       8                     4
Analytics and Computational Modelling   0.500000       False                       0                     0
       Asset and Liability Management   0.500000       False                       0                     0
                 Attribution Analysis   0.500000       False                       0                     0
                     Audit Compliance   0.472885        True                      18                     6
           

In [ ]:
comparison_df = pd.DataFrame([
    {
        "model_variant": "ovr_global_threshold",
        "threshold_type": "global",
        "micro_precision": global_precision,
        "micro_recall": global_recall,
        "micro_f1": global_f1,
        "global_threshold": best_threshold,
    },
    {
        "model_variant": "ovr_skill_specific_threshold",
        "threshold_type": "skill_specific",
        "micro_precision": skill_precision,
        "micro_recall": skill_recall,
        "micro_f1": skill_f1,
        "global_threshold": np.nan,
    },
])

print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

               model_variant threshold_type  micro_precision  micro_recall  micro_f1  global_threshold
        ovr_global_threshold         global           0.7007        0.8276    0.7589            0.5056
ovr_skill_specific_threshold skill_specific           0.1536        0.9310    0.2637               NaN


In [ ]:
def indicator_row_to_skills(row, skill_names):
    return [skill_names[j] for j, flag in enumerate(row) if flag == 1]

global_predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in global_val_pred]
actual_skill_lists_val = jobs_val_df["actual_skill_lists"].tolist()

global_predictions_df = pd.DataFrame({
    "job_id": jobs_val_df["job_id"],
    "title": jobs_val_df["title_label"],
    "actual_skills": [" | ".join(skills) for skills in actual_skill_lists_val],
    "predicted_skills": [" | ".join(skills) for skills in global_predicted_skill_lists],
    "tp": [len(set(a) & set(p)) for a, p in zip(actual_skill_lists_val, global_predicted_skill_lists)],
    "fp": [len(set(p) - set(a)) for a, p in zip(actual_skill_lists_val, global_predicted_skill_lists)],
    "fn": [len(set(a) - set(p)) for a, p in zip(actual_skill_lists_val, global_predicted_skill_lists)],
})

print(global_predictions_df.head(10).to_string(index=False))

                          job_id                                                                      title                                                                                                                                                        actual_skills                                                                                                                                                                                                                              predicted_skills  tp  fp  fn
32df38ce68ced2f5ade19285fd7d9413                                                      Junior Auditor - YZ11   Auditing and Assurance Standards | Engagement Planning | Engagement Execution | Internal Controls | Professional and Business Ethics | Financial Statements Review Auditing and Assurance Standards | Engagement Completion and Reporting | Engagement Execution | Engagement Planning | Financial Statements Review | Information Gathering and Analysis | Internal Controls |

In [ ]:
global_macro_precision, global_macro_recall, global_macro_f1, _ = precision_recall_fscore_support(
    Y_val,
    global_val_pred,
    average="macro",
    zero_division=0,
)

skill_macro_precision, skill_macro_recall, skill_macro_f1, _ = precision_recall_fscore_support(
    Y_val,
    skill_val_pred,
    average="macro",
    zero_division=0,
)

comparison_df = pd.DataFrame([
    {
        "model_variant": "ovr_global_threshold",
        "threshold_type": "global",
        "micro_precision": global_precision,
        "micro_recall": global_recall,
        "micro_f1": global_f1,
        "macro_precision": global_macro_precision,
        "macro_recall": global_macro_recall,
        "macro_f1": global_macro_f1,
        "global_threshold": best_threshold,
    },
    {
        "model_variant": "ovr_skill_specific_threshold",
        "threshold_type": "skill_specific",
        "micro_precision": skill_precision,
        "micro_recall": skill_recall,
        "micro_f1": skill_f1,
        "macro_precision": skill_macro_precision,
        "macro_recall": skill_macro_recall,
        "macro_f1": skill_macro_f1,
        "global_threshold": np.nan,
    },
])

print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

               model_variant threshold_type  micro_precision  micro_recall  micro_f1  macro_precision  macro_recall  macro_f1  global_threshold
        ovr_global_threshold         global           0.7007        0.8276    0.7589           0.0601        0.0716    0.0624            0.5056
ovr_skill_specific_threshold skill_specific           0.1536        0.9310    0.2637           0.0809        0.0916    0.0807               NaN


## Summary of OVR Threshold Results

This notebook implemented a threshold-based one-vs-rest logistic regression baseline for ACC skill tagging using job embeddings from `embedding/acc`.

**Global threshold:**

Use one cutoff for every skill.
Example: predict a skill if probability >= 0.5056 for all skills.
Same rule everywhere.

**Skill-specific threshold:**

Each skill gets its own cutoff.
Example:
Accounting Standards >= 0.52
Audit Compliance >= 0.47
Auditing and Assurance Standards >= 0.61

### Main takeaway
The global-threshold OVR model performed much better than the skill-specific-threshold version on the validation set.

### Validation results
- `ovr_global_threshold`
  - Micro Precision: `0.7007`
  - Micro Recall: `0.8276`
  - Micro F1: `0.7589`
  - Macro Precision: `0.0601`
  - Macro Recall: `0.0716`
  - Macro F1: `0.0624`
  - Best global threshold: `0.5056`

- `ovr_skill_specific_threshold`
  - Micro Precision: `0.1536`
  - Micro Recall: `0.9310`
  - Micro F1: `0.2637`
  - Macro Precision: `0.0809`
  - Macro Recall: `0.0916`
  - Macro F1: `0.0807`

### Interpretation
The global-threshold model is the stronger OVR threshold competitor in this setup. It gives a good balance between precision and recall and produces sensible predictions for common accounting skills.

The skill-specific-threshold version appears to overfit because the validation set is small and many skills have very limited support. This causes recall to become very high, but precision collapses.

### Practical conclusion
For the threshold-based OVR competitor, the global-threshold version should be treated as the main benchmark result from this notebook.